In [3]:
import pandas as pd
import re
from IPython.display import display

In [24]:
filename = 'Final_DatasetB_102725.xlsx'
df = pd.read_excel(filename)

# Print the total number of rows in the dataset
total_rows = len(df)
print(f"Total number of rows in the dataset: {total_rows}")

VALID_TLDS = {
    'com', 'org', 'net', 'gov', 'edu', 'info', 'io', 'co', 'us', 'uk', 'de', 'fr', 'ca', 'au', 'ch', 'jp', 'cn'
}

def is_valid_domain(domain):
    # Must contain at least one dot
    if '.' not in domain:
        return False
    # Extract TLD and check validity
    tld = domain.split('.')[-1].lower()
    if tld not in VALID_TLDS:
        return False
    # Should not contain social media or other non-domain words
    invalid_words = {'twitter', 'facebook', 'linkedin', 'instagram'}
    for word in invalid_words:
        if word in domain.lower():
            return False
    # No spaces allowed
    if ' ' in domain:
        return False
    # Reasonable length
    if len(domain) < 5 or len(domain) > 50:
        return False
    return True

def is_valid_name(first, last):
    if not (first and last):
        return False
    if len(first) < 2 or len(last) < 2:
        return False
    if not (first[0].isupper() and last[0].isupper()):
        return False
    
    # Expanded list of invalid words
    invalid_words = set([
        'At', 'To', 'From', 'Cc', 'Or', 'Email', 'Sent', 'Drop', 'Forwarded', 'Subject',
        'The', 'Post', 'News', 'Office', 'Press', 'Team', 'Group', 'Department'
    ])
    
    # Check if first or last name matches invalid words
    if first in invalid_words or last in invalid_words:
        return False
    
    # Additional pattern checks for non-person names (e.g., "The Post")
    if first.lower() == "the" or last.lower() in {"post", "news", "office", "team", "group"}:
        return False
    
    return True

def extract_people(text):
    results = []
    for line in str(text).splitlines():
        if line.strip().lower().startswith(('to:', 'cc:')):
            pattern = r'([A-Z][a-zA-Z\'\-]+(?:[, ]\s*[A-Z][a-zA-Z\'\-]+)+)[<\s]*[█\w\.-]*@([a-zA-Z0-9\.-]+\.[a-zA-Z]{2,})'
            matches = re.findall(pattern, line)
            for raw_name, domain in matches:
                name = raw_name.strip().replace('"', '')
                
                # ---- existing name parsing ----
                if ',' in name:
                    parts = [x.strip() for x in name.split(',', 1)]
                    if len(parts) == 2:
                        last, first = parts
                    else:
                        continue
                else:
                    parts = name.split()
                    if len(parts) == 2:
                        first, last = parts
                    elif len(parts) > 2:
                        first = parts[0]
                        last = parts[-1]
                    else:
                        continue

                # ---- CLEANING STEP ----
                remove_list = ["AM", "PM", "´"]
                for rem in remove_list:
                    first = first.replace(rem, "")
                    last = last.replace(rem, "")

                first = first.strip()
                last = last.strip()

                 # This fixes: Brett Clanton, Phil Cochrane, Joe Ellis, Allison Grissel, etc.
                if first in {"Brett", "Phil", "Joe", "Allison", "Laura", "Geoff", "James", "Elizabeth", "Robert", "Mary", "Suzanne"}:
                    first, last = last, first  # swap them

                # ---- DELETION RULES ----
                delete_first = {
                    "Integrated Communications", "Scientific", "Federal",
                    "Executive", "BGOV", "BP", "DC", "External", "Hop",
                    "Jame", "Client RelationsDDC", "POLITICO", "International",
                    "Clean", "Communications", "Upstream", "Resources", "AM"
                }

                delete_last = {
                    "AM", "Government", "Affairs", "Information",
                    "CA Report", "Institution"
                }

                if first in delete_first or last in delete_last:
                    continue

                # ---- validation + append ----
                if is_valid_name(first, last) and is_valid_domain(domain):
                    full_name = f"{first} {last}"
                    results.append({
                        'Name': first,
                        'Lastname': last,
                        'Full Name': full_name,
                        'Company': domain
                    })
    return results


# Extract valid names and domains
all_people = []
for text in df['File Content']:
    all_people.extend(extract_people(text))

# Create DataFrame from extracted people
people_df = pd.DataFrame(all_people, columns=['Name', 'Lastname', 'Full Name', 'Company'])

# Print the number of valid names found
valid_names_count = len(people_df)
print(f"Number of valid names found: {valid_names_count}")

# Convert Company names to lowercase for case-insensitive sorting
people_df['Company'] = people_df['Company'].str.lower()

# Sort first by Company, then by Name
people_df_sorted = people_df.sort_values(by=['Company', 'Name'])

# Remove duplicates
people_df_sorted = people_df_sorted.drop_duplicates(subset=['Name', 'Lastname', 'Company'])

def clean_company(domain):
    parts = domain.split('.')
    # remove last element (TLD)
    parts = parts[:-1]
    # uppercase and join with spaces
    return " ".join([p.upper() for p in parts])

people_df_sorted['Company_clean'] = people_df_sorted['Company'].apply(clean_company)

# Print the number of rows after removing duplicates
unique_names_count = len(people_df_sorted)
print(f"Number of valid names after removing duplicates: {unique_names_count}")

Total number of rows in the dataset: 4479
Number of valid names found: 1821
Number of valid names after removing duplicates: 479


In [27]:
people_df_sorted['Company_clean'].value_counts().head(15)

Company_clean
BP                307
UK BP              47
API                14
NPC                11
SE1 BP             10
EXXONMOBIL          9
BPX                 5
GMAIL               5
AXIOS               3
OGILVYGR            3
BRUNSWICKGROUP      3
PODESTA             3
ALLIANCESNW         3
CLCOUNCIL           3
PODESTAGROUP        2
Name: count, dtype: int64

In [30]:
energy_industry = {"BP", "UK BP", "SE1 BP", "API", "EXXONMOBIL"}
lobby = {"NPC"}
media = {"AXIOS"}

def assign_category(company):
    if company in energy_industry:
        return "Energy Industry"
    if company in lobby:
        return "Lobby"
    if company in media:
        return "Media"
    return ""

people_df_sorted["NER_Category"] = people_df_sorted["Company_clean"].apply(assign_category)

In [31]:
# Save to CSV
output_filename = 'Extracted_email_names_datasetB_for_validation.csv'
people_df_sorted.to_csv(output_filename, index=False)

print(f"Saved sorted, deduplicated results to {output_filename}")

# Display as a nice table in Jupyter
display(people_df_sorted)

Saved sorted, deduplicated results to Extracted_email_names_datasetB_for_validation.csv


,Name,Lastname,Full Name,Company,Company_clean,NER_Category
121,Geoffrey,Moody,Geoffrey Moody,afpm.org,AFPM,
966,Denny,Eliason,Denny Eliason,alliancesnw.com,ALLIANCESNW,
1616,Denny,Eliaso,Denny Eliaso,alliancesnw.com,ALLIANCESNW,
967,Kim,Clauson,Kim Clauson,alliancesnw.com,ALLIANCESNW,
365,Jim,Massie,Jim Massie,alpinegroup.com,ALPINEGROUP,
...,...,...,...,...,...,...
945,Spencer,Dale,Spencer Dale,uk.bp.com,UK BP,Energy Industry
703,Stout,Robert,Stout Robert,uk.bp.com,UK BP,Energy Industry
1308,Wendy,Lindskoog,Wendy Lindskoog,uk.bp.com,UK BP,Energy Industry
1106,Yogaananda,Maharaj,Yogaananda Maharaj,uk.bp.com,UK BP,Energy Industry


# Now for Dataset A 

In [32]:
filename = 'Final_DatasetA_102725.xlsx'
df = pd.read_excel(filename)

# Print the total number of rows in the dataset
total_rows = len(df)
print(f"Total number of rows in the dataset: {total_rows}")

def is_valid_name(first, last):
    if not (first and last):
        return False
    if len(first) < 2 or len(last) < 2:
        return False
    if not (first[0].isupper() and last[0].isupper()):
        return False
    
    # Expanded list of invalid words
    invalid_words = set([
        'At', 'To', 'From', 'Cc', 'Or', 'Email', 'Sent', 'Drop', 'Forwarded', 'Subject',
        'The', 'Post', 'News', 'Office', 'Press', 'Team', 'Group', 'Department'
    ])
    
    # Check if first or last name matches invalid words
    if first in invalid_words or last in invalid_words:
        return False
    
    # Additional pattern checks for non-person names (e.g., "The Post")
    if first.lower() == "the" or last.lower() in {"post", "news", "office", "team", "group"}:
        return False
    
    return True

def extract_people(text):
    results = []
    for line in str(text).splitlines():
        if line.strip().lower().startswith(('to:', 'cc:')):
            pattern = r'([A-Z][a-zA-Z\'\-]+(?:[, ]\s*[A-Z][a-zA-Z\'\-]+)+)[<\s]*[█\w\.-]*@([a-zA-Z0-9\.-]+\.[a-zA-Z]{2,})'
            matches = re.findall(pattern, line)
            for raw_name, domain in matches:
                name = raw_name.strip().replace('"', '')
                
                # ---- existing name parsing ----
                if ',' in name:
                    parts = [x.strip() for x in name.split(',', 1)]
                    if len(parts) == 2:
                        last, first = parts
                    else:
                        continue
                else:
                    parts = name.split()
                    if len(parts) == 2:
                        first, last = parts
                    elif len(parts) > 2:
                        first = parts[0]
                        last = parts[-1]
                    else:
                        continue

                # ---- CLEANING STEP ----
                remove_list = ["AM", "PM", "´"]
                for rem in remove_list:
                    first = first.replace(rem, "")
                    last = last.replace(rem, "")

                first = first.strip()
                last = last.strip()

                 # This fixes: Brett Clanton, Phil Cochrane, Joe Ellis, Allison Grissel, etc.
                if first in {"Brett", "Phil", "Joe", "Allison", "Laura", "Geoff", "James", "Elizabeth", "Robert", "Mary", "Suzanne"}:
                    first, last = last, first  # swap them

                # ---- DELETION RULES ----
                delete_first = {
                    "Integrated Communications", "Scientific", "Federal",
                    "Executive", "BGOV", "BP", "DC", "External", "Hop",
                    "Jame", "Client RelationsDDC", "POLITICO", "International",
                    "Clean", "Communications", "Upstream", "Resources", "AM"
                }

                delete_last = {
                    "AM", "Government", "Affairs", "Information",
                    "CA Report", "Institution"
                }

                if first in delete_first or last in delete_last:
                    continue

                # ---- validation + append ----
                if is_valid_name(first, last) and is_valid_domain(domain):
                    full_name = f"{first} {last}"
                    results.append({
                        'Name': first,
                        'Lastname': last,
                        'Full Name': full_name,
                        'Company': domain
                    })
    return results


# Extract valid names and domains
all_people = []
for text in df['File Content']:
    all_people.extend(extract_people(text))

# Create DataFrame from extracted people
people_df = pd.DataFrame(all_people, columns=['Name', 'Lastname', 'Full Name', 'Company'])

# Print the number of valid names found
valid_names_count = len(people_df)
print(f"Number of valid names found: {valid_names_count}")

# Convert Company names to lowercase for case-insensitive sorting
people_df['Company'] = people_df['Company'].str.lower()

# Sort first by Company, then by Name
people_df_sorted = people_df.sort_values(by=['Company', 'Name'])

# Remove duplicates
people_df_sorted = people_df_sorted.drop_duplicates(subset=['Name', 'Lastname', 'Company'])

def clean_company(domain):
    parts = domain.split('.')
    # remove last element (TLD)
    parts = parts[:-1]
    # uppercase and join with spaces
    return " ".join([p.upper() for p in parts])

people_df_sorted['Company_clean'] = people_df_sorted['Company'].apply(clean_company)

# Print the number of rows after removing duplicates
unique_names_count = len(people_df_sorted)
print(f"Number of valid names after removing duplicates: {unique_names_count}")

Total number of rows in the dataset: 218
Number of valid names found: 34
Number of valid names after removing duplicates: 22


In [33]:
people_df_sorted['Company_clean'].value_counts().head(15)

Company_clean
BP                3
NAM               3
API               2
HKSTRATEGIES      2
IPAA              2
AGA               1
BRUNSWICKGROUP    1
COLUMBIA          1
DVCN              1
EXXONMOBIL        1
INGAA             1
MCKINSEY          1
NEWFIELD          1
NGSA              1
NPC               1
Name: count, dtype: int64

In [34]:
energy_industry = {"BP", "UK BP", "SE1 BP", "API", "EXXONMOBIL"}
lobby = {"NPC"}
media = {"AXIOS"}

def assign_category(company):
    if company in energy_industry:
        return "Energy Industry"
    if company in lobby:
        return "Lobby"
    if company in media:
        return "Media"
    return ""

people_df_sorted["NER_Category"] = people_df_sorted["Company_clean"].apply(assign_category)

In [36]:
# Save to CSV
output_filename = 'Extracted_email_names_datasetA_for_validation.csv'
people_df_sorted.to_csv(output_filename, index=False)

print(f"Saved sorted, deduplicated results to {output_filename}")

# Display as a nice table in Jupyter
display(people_df_sorted)

Saved sorted, deduplicated results to Extracted_email_names_datasetA_for_validation.csv


,Name,Lastname,Full Name,Company,Company_clean,NER_Category
8,Jen,O'Shea,Jen O'Shea,aga.org,AGA,
3,Barrus,Brett,Barrus Brett,api.org,API,Energy Industry
2,Sari,Fink,Sari Fink,api.org,API,Energy Industry
17,Ellis,Joe,Ellis Joe,bp.com,BP,Energy Industry
24,Jason,Ryan,Jason Ryan,bp.com,BP,Energy Industry
20,Sally,Kolenda,Sally Kolenda,bp.com,BP,Energy Industry
11,Ellen,Moskowitz,Ellen Moskowitz,brunswickgroup.com,BRUNSWICKGROUP,
19,Susanne,Rust,Susanne Rust,columbia.edu,COLUMBIA,
0,Bill,Green,Bill Green,dvcn.com,DVCN,
21,Nick,Schulz,Nick Schulz,exxonmobil.com,EXXONMOBIL,Energy Industry


# Final Part of Harmonization with Individuals/Persons Dataset

In [130]:
# Josh manually cleaned and completed the harmonized list of "email individuals", then Sattiki finalized the task in Feb 2026.
# Go on working from this, open the file with the extracted email names and merge it with ner_person_complete. 
# Check how many duplicates we have first and then add the new individuals to the list. Then do desk reseearch to complete the meta data vars.
# We call the list of individuals with meta info "target" 

import pandas as pd 
import re 
from IPython.display import display 
from pathlib import Path

filename = r"C:\Users\charlott\Dropbox (Personal)\CSSN_Team_Folder\Data\NER_email_extraction\Extracted_email_names_final.xlsx"

df = pd.read_excel(filename)
df = df.iloc[:, :6]
print(df.columns.tolist())
dup_fullname_rows = df["Full Name"].duplicated().sum()
print("Duplicate rows in df (Full Name):", dup_fullname_rows)

Rows: 555
['Entities_filtered', 'Role', 'Employee_level', 'Workplace', 'Industry_broad', 'Industry_specific']
Duplicate rows in df (Entities_filtered): 0


In [16]:
# --- load target_df ---
target_path = Path(r"C:\Users\charlott\Dropbox (Personal)\Paper_Corporate_Obstructionism\Analysis\B_NER\Preparation_Database_Publication\ner_person_complete.xlsx")

target_df = pd.read_excel(target_path)
print("target_df columns:", target_df.columns.tolist())
print("Rows before:", len(target_df))
target_df = target_df[target_df["Industry_broad"] != "drop"].copy()
print("Rows remaining when dropping the ones to drop:", len(target_df))

# --- counts: duplicates + new ones (relative to target) ---
# duplicates in df by Full Name

# duplicates in target by Entities_filtered (useful diagnostic)
dup_target_rows = target_df["Entities_filtered"].duplicated().sum()

# new names in df that are NOT in target (count as unique entities, not rows)
df_unique_names = df["Full Name"].dropna().drop_duplicates()
target_unique_names = target_df["Entities_filtered"].dropna().drop_duplicates()

new_unique_entities_added = (~df_unique_names.isin(target_unique_names)).sum()

print("Duplicate rows in target_df (Entities_filtered):", dup_target_rows)
print("New unique entities in df not in target_df:", new_unique_entities_added)


# 1) Build an "append_df" from df in the same schema as target_df
append_df = pd.DataFrame({
    "Entities_filtered": df["Full Name"],
    "Workplace": df["Company_clean"],
    "Industry_broad": df["NER_Category"]
})

# keep only valid names
append_df = append_df.dropna(subset=["Entities_filtered"])

# 2) Remove entities that already exist in target_df (so we only append NEW ones)
existing = set(target_df["Entities_filtered"].dropna())
append_df_new = append_df[~append_df["Entities_filtered"].isin(existing)].copy()

print("Rows to append:", len(append_df_new))  # should be 379

# 3) Append
target_df_updated = pd.concat([target_df, append_df_new], ignore_index=True)

print("names codebook before:", len(target_df))
print("names codebook after :", len(target_df_updated))
print("Hence, added rows   :", len(target_df_updated) - len(target_df))

target_df columns: ['Entities_filtered', 'Role', 'Workplace', 'Industry_broad', 'Industry_specific', 'Unnamed: 5']
Rows before: 227
Rows remaining when dropping the ones to drop: 193
Duplicate rows in target_df (Entities_filtered): 0
New unique entities in df not in target_df: 363
Rows to append: 363
names codebook before: 193
names codebook after : 556
Hence, added rows   : 363


In [5]:
target_df_updated = target_df_updated.drop_duplicates(subset=["Entities_filtered"], keep="first")
print("full length of names codebook after deduplication:", len(target_df_updated))

full length of names codebook after deduplication: 556


In [6]:
target_df_updated[target_df_updated["Entities_filtered"].str.startswith("John", na=False)]["Entities_filtered"]

76           John Barrasso
77             John Chafee
78     John Jacob Astor IV
79              John James
80            John Kennedy
81          John Kennedy's
82            John Podesta
83           John S Dryzek
84             John Tonaki
85        Johnjerica Hodge
233            John Wagner
330         John Phillippe
331          John D'Andrea
332            John Harvey
456         John M' Dabbar
487               John Guy
509         John MacArthur
Name: Entities_filtered, dtype: object

In [7]:
target_df_updated[target_df_updated["Entities_filtered"].str.startswith("Mark", na=False)]["Entities_filtered"]

106      Mark Amodei
360    Mark Borowski
361      Mark Stultz
362       Mark Bunch
363      Mark Finley
364      Mark Sutter
481     Mark Denzler
Name: Entities_filtered, dtype: object

In [8]:
target_df_updated[target_df_updated["Entities_filtered"].str.startswith("Michael", na=False)]["Entities_filtered"]

112     Michael Franczak
113         Michael Mann
114       Michael Ratner
115        Michael Regan
116     Michael S. Regan
373        Michael Cohen
374      Michael Wortham
375        Michael Brien
376    Michael Abendhoff
475        Michael Steel
549         Michael Kehs
Name: Entities_filtered, dtype: object

In [9]:
print(target_df_updated.columns.tolist())
target_df_updated = target_df_updated.iloc[:, :5]

['Entities_filtered', 'Role', 'Workplace', 'Industry_broad', 'Industry_specific', 'Unnamed: 5']


In [10]:
n_entities = target_df_updated["Entities_filtered"].nunique(dropna=True)
n_entities

556

In [11]:
entities_per_industry = (
    target_df_updated
    .dropna(subset=["Entities_filtered", "Industry_broad"])
    .groupby("Industry_broad")["Entities_filtered"]
    .nunique()
    .sort_values(ascending=False)
)

entities_per_industry

Industry_broad
Energy Industry                304
Politician                      65
Academia                        49
Lobby                           30
Other Industry                  20
NGO/Thinktank/Foundation        20
Government Agency               17
PR Company                      13
Keyword Climate Environment     12
Consultancy                     10
Consultancy\t                    3
Media                            2
Journalist                       2
Litigation                       1
API                              1
International Organization       1
News Outlet                      1
Energy/Industry                  1
Academia,Politician              1
\tConsultancy                    1
Name: Entities_filtered, dtype: int64

In [12]:
output_filename = 'ner_person_complete_including_emails.csv'
target_df_updated.to_csv(output_filename, index=False)

print(f"Saved to {output_filename}")

# Display as a nice table in Jupyter
display(target_df_updated)

Saved to ner_person_complete_including_emails.csv


,Entities_filtered,Role,Workplace,Industry_broad,Industry_specific
0,Alan Jeffers,Retired communicator at Exxon Mobil Corporation,"\nExxonMobil Corporation,Texas ,USA",Energy Industry,"Public Affairs, Media Relations"
1,Alexander V. Mirtchev,"U.S. academic, business executive, and author,...","George Mason University, USA",Academia,international economics and energy policies
2,Alicia Colomer,Managing Director at Campus Climate Network,Campus Climate Network,NGO/Thinktank/Foundation,Climate and Sustainability
3,Andrea Woods,Senior Media Relations Manager at the American...,Washington DC-Baltimore Area,Lobby,"Media Relations, Public Affairs"
4,Andrew Wheeler,United States Environmental Protection Agency ...,"U.S. Environmental Protection Agency (EPA), 12...",Politician,Environmental Policy and Regulation
...,...,...,...,...,...
551,Jeff Eshelman,NaN,IPAA,Lobby,NaN
552,Kassia Yanosek,NaN,MCKINSEY,Consultancy,NaN
553,Brittany Seabury,NaN,NAM,Lobby,NaN
554,Jeff Pierce,NaN,NAM,Lobby,NaN


Give this to the RA to manually add info for columns "Role" and "Industry Specific"

###  Now harmonize ner_person_complete_including_emails with the NER codebook
the ner_person_complete_including_emails has 556 rows, while the codebook has 1004

In [19]:
codebook_path = Path(r"C:\Users\charlott\Dropbox (Personal)\Paper_Corporate_Obstructionism\Analysis\B_NER\Preparation_Database_Publication\NER_Codebook_1004.xlsx")

codebook = pd.read_excel(codebook_path)
print("codebook_path columns:", codebook.columns.tolist())
print("Rows:", len(codebook))

print(sorted(codebook["Sub_category"].dropna().unique()))

codebook_path columns: ['Entities_filtered', 'Sub_category']
Rows: 1004
['Academia', 'Consultancy', 'Courts', 'Energy Industry', 'Government Agency', 'International Climate Governance', 'International Organisation', 'Journalist', 'Keyword Climate Environment', 'Keyword Obstruction', 'Legal Professionals', 'Legislation', 'Litigation', 'Lobby', 'NGO/Thinktank/Foundation', 'News Outlet', 'Other Industry', 'PR Company', 'Politician']


In [20]:
# now save the non - duplicates (new individuals) in a separate df because we need it for appending later 

# --- A) build NON-duplicates (i.e., entities NOT in the overlap) ---

# sets of entities
codebook_entities = set(codebook["Entities_filtered"].dropna())
target_entities   = set(target_df_updated["Entities_filtered"].dropna())

# non-overlapping entities
codebook_only = codebook_entities - overlap_entities
target_only   = target_entities - overlap_entities

print(f"Codebook-only entities: {len(codebook_only)}")
print(f"Target-only entities:   {len(target_only)}")

# subset both datasets to non-duplicates
codebook_nondup = codebook.loc[
    codebook["Entities_filtered"].isin(codebook_only)
].copy()

target_nondup = target_df_updated.loc[
    target_df_updated["Entities_filtered"].isin(target_only)
].copy()

# harmonize (same logic you used above)
codebook_nondup["Industry_harmonized"] = codebook_nondup["Sub_category"]

codebook_nondup = codebook_nondup.drop(columns=["Sub_category"], errors="ignore")
target_nondup   = target_nondup.drop(columns=["Industry_broad"], errors="ignore")

# add source indicator
codebook_nondup["Source_dataset"] = "codebook"
target_nondup["Source_dataset"]   = "target_df_updated"

# align columns (union)
all_cols_nondup = sorted(set(codebook_nondup.columns) | set(target_nondup.columns))
codebook_nondup = codebook_nondup.reindex(columns=all_cols_nondup)
target_nondup   = target_nondup.reindex(columns=all_cols_nondup)

# combine non-duplicates
non_duplicates_full = pd.concat([codebook_nondup, target_nondup], ignore_index=True)

# check whether names are among NON-duplicates
names_to_check = {"Sarah Dimond", "Chris Lockett"}
present = names_to_check & set(non_duplicates_full["Entities_filtered"].dropna())
missing = names_to_check - present

print("Among non-duplicates (present):", sorted(present))
print("Not found among non-duplicates:", sorted(missing))

# (optional) export
non_duplicates_full.to_excel("non_duplicates_full.xlsx", index=False)
print("Saved: non_duplicates_full.xlsx")

NameError: name 'overlap_entities' is not defined

In [259]:
# --- B) transform a dataset: Full Name -> Entities_filtered, NER_Category -> Sub_category ---
# If the dataset you mean is non_duplicates_full, use it here.
# If you mean a different dataframe, replace `non_duplicates_full` with that df.

email_individuals = non_duplicates_full.rename(
    columns={"Full Name": "Entities_filtered", "NER_Category": "Sub_category"}
).copy()

# sanity check: show whether the renamed columns exist now
print("email_individuals columns:", email_individuals.columns.tolist())
email_individuals = email_individuals.dropna(subset=["Entities_filtered", "Sub_category"])
print(len(email_individuals))

email_individuals columns: ['Entities_filtered', 'Industry_harmonized', 'Industry_specific', 'Role', 'Source_dataset', 'Workplace']


KeyError: ['Sub_category']

In [232]:
print(list(overlap_entities)[:10])

['Rep Raskin', 'Roishetta Sibley Ozane', 'Christian Downie', 'Raul Ruiz', 'Chuck Schumer', 'Michael Regan', 'Karl Marx', 'Jared Diamond', 'Pete Stauber', 'Ronald Johnson']


In [4]:
# remove later, just to read it back in 
duplicates_full = pd.read_excel("overlap_duplicates_full.xlsx")
duplicates_full = duplicates_full.drop(columns=["new"], errors="ignore")
print(duplicates_full.columns.tolist())

['Entities_filtered', 'Industry_harmonized', 'Industry_specific', 'Role', 'Source_dataset', 'Workplace']


In [5]:
# additional preprocessing 
str_cols = duplicates_full.select_dtypes(include="object").columns
duplicates_full[str_cols] = duplicates_full[str_cols].apply(lambda c: c.str.strip())

def _normalize_entity_name(x):
    if pd.isna(x):
        return x
    s = str(x).strip()

    # 1) remove trailing possessive: "John Kennedy's" / "John Kennedy’s" -> "John Kennedy"
    s = re.sub(r"(?:'s|’s)\s*$", "", s).strip()

    # 2) collapse weird whitespace (tabs/newlines) and strip again
    s = re.sub(r"\s+", " ", s).strip()

    # 3) no capslock: convert to Title Case (keeps roman numerals as-is later)
    s = s.title()

    # fix common roman numerals that title() would lower-case
    s = re.sub(r"\bIii\b", "III", s)
    s = re.sub(r"\bIi\b", "II", s)
    s = re.sub(r"\bIv\b", "IV", s)
    s = re.sub(r"\bVi\b", "VI", s)
    s = re.sub(r"\bVii\b", "VII", s)
    s = re.sub(r"\bViii\b", "VIII", s)
    s = re.sub(r"\bIx\b", "IX", s)

    return s

# Step A: base normalization
duplicates_full["Entities_filtered"] = duplicates_full["Entities_filtered"].apply(_normalize_entity_name)

# Step B: "keep the longer name if the shorter is contained in it"
# Build a mapping from shorter -> longer among names present
names = duplicates_full["Entities_filtered"].dropna().astype(str).unique().tolist()

# Sort by length descending so we prefer longer canonical names
names_sorted = sorted(names, key=len, reverse=True)

mapping = {}
for long_name in names_sorted:
    for short_name in names_sorted[::-1]:  # shortest first
        if short_name == long_name:
            continue
        # containment rule: shorter contained in longer (case-insensitive, whole substring)
        if short_name.lower() in long_name.lower():
            # only map if we haven't already assigned a longer canonical target
            if short_name not in mapping or len(long_name) > len(mapping[short_name]):
                mapping[short_name] = long_name

# Apply mapping
duplicates_full["Entities_filtered"] = duplicates_full["Entities_filtered"].replace(mapping)

In [6]:
# Case 1: Same Entities_filtered, same Industry_harmonized, one has "Workplace" empty, the other one is not empty. In this case, deduplicate by keeping the row with no empty columns. To indicate that the entitiy is coming from two sources, indicate both sources in "Source_dataset". 
# So "codebook; target_df_updated" 

def _is_empty_series(s: pd.Series) -> pd.Series:
    """Treat NaN, empty string, or whitespace-only as empty."""
    return s.isna() | s.astype(str).str.strip().eq("")

def _non_empty_count_row(row: pd.Series) -> int:
    """Count non-empty cells in a row (excluding source_dataset for scoring)."""
    row = row.drop(labels=["source_dataset"], errors="ignore")
    empties = row.isna() | row.astype(str).str.strip().eq("")
    return int((~empties).sum())

def dedup_case_1(df: pd.DataFrame) -> pd.DataFrame:
    key_cols = ["Entities_filtered", "Industry_harmonized"]

    if "Workplace" not in df.columns:
        return df

    def handle_group(g: pd.DataFrame) -> pd.DataFrame:
        # Re-attach the grouping columns because include_groups=False removes them
        # g.name is a tuple in the same order as key_cols
        if isinstance(g.name, tuple):
            for col, val in zip(key_cols, g.name):
                g[col] = val
        else:
            # in case you ever group by a single key
            g[key_cols[0]] = g.name

        if len(g) < 2:
            return g

        workplace_empty = _is_empty_series(g["Workplace"])
        has_empty = workplace_empty.any()
        has_non_empty = (~workplace_empty).any()

        if has_empty and has_non_empty:
            sources = []
            for v in g.get("source_dataset", pd.Series([], dtype=str)).astype(str).tolist():
                parts = [p.strip() for p in v.split(";") if p.strip()]
                sources.extend(parts)
            merged_sources = "; ".join(sorted(dict.fromkeys(sources)))

            scores = g.apply(_non_empty_count_row, axis=1)
            keep_idx = scores.idxmax()

            kept = g.loc[[keep_idx]].copy()
            kept.loc[:, "source_dataset"] = merged_sources
            return kept

        return g

    out = (
        df.groupby(key_cols, dropna=False, sort=False, group_keys=False)
          .apply(handle_group, include_groups=False)
          .reset_index(drop=True)
    )
    return out

# Apply Case 1 deduplication
duplicates_full = dedup_case_1(duplicates_full)

In [7]:
print(len(duplicates_full))
print([repr(c) for c in duplicates_full.columns])

print(
    duplicates_full
    .sort_values("Entities_filtered")
    [["Entities_filtered", "Workplace", "Source_dataset"]]
    .head(8)
)

# this worked well. "Angus Berwick" is e.g. still in there twice cause Column B differs from one source to the other.

398
["'Industry_specific'", "'Role'", "'Source_dataset'", "'Workplace'", "'Entities_filtered'", "'Industry_harmonized'"]
         Entities_filtered                          Workplace  \
209           Alan Jeffers  ExxonMobil Corporation,Texas ,USA   
66            Alan Jeffers                                NaN   
0    Alexander V. Mirtchev                                NaN   
129  Alexander V. Mirtchev                                NaN   
52   Alexander V. Mirtchev                                NaN   
210  Alexander V. Mirtchev       George Mason University, USA   
117         Alicia Colomer                                NaN   
211         Alicia Colomer             Campus Climate Network   

        Source_dataset  
209  target_df_updated  
66            codebook  
0             codebook  
129           codebook  
52            codebook  
210  target_df_updated  
117           codebook  
211  target_df_updated  


In [8]:
duplicates_full = duplicates_full.replace(
    {"International Organization": "International Organisation"}
)
duplicates_full.loc[
    (duplicates_full["Entities_filtered"] == "Ernest Moniz") &
    (duplicates_full["Industry_harmonized"] == "Academia,Politician"),
    "Industry_harmonized"
] = "Politician"
print(sorted(duplicates_full["Industry_harmonized"].dropna().unique()))

['Academia', 'Consultancy', 'Energy Industry', 'Government Agency', 'International Organisation', 'Journalist', 'Keyword Climate Environment', 'Legal Professionals', 'Lobby', 'NGO/Thinktank/Foundation', 'Other Industry', 'PR Company', 'Politician']


In [9]:
# Case 2: The Entities_filtered column is identical but Industry_harmonized differs. # 
# All rows except one (for the identical Entities_filtered) are also empty in the "Industry_specific" column. If this is the case, then add the different values of "Industry_harmonized" together. 
# For instance "Government Agency; Politician". then: remove "Other Industry" from the combined column if is it not the only entry in the column.

def _combine_industries(values) -> str:
    # normalize, split on ';' (in case already combined), unique, keep order
    items = []
    for v in values:
        if pd.isna(v):
            continue
        for part in str(v).split(";"):
            p = part.strip()
            if p:
                items.append(p)
    # unique preserving order
    seen = set()
    uniq = []
    for x in items:
        if x not in seen:
            uniq.append(x)
            seen.add(x)

    # remove "Other Industry" if it's not the only entry
    if "Other Industry" in seen and len(uniq) > 1:
        uniq = [x for x in uniq if x != "Other Industry"]

    return "; ".join(uniq)

def dedup_case_2(df: pd.DataFrame) -> pd.DataFrame:
    if "Entities_filtered" not in df.columns or "Industry_harmonized" not in df.columns:
        return df
    if "Industry_specific" not in df.columns:
        return df

    def handle_entity(g: pd.DataFrame) -> pd.DataFrame:
        # re-attach grouping col because include_groups=False strips it
        g["Entities_filtered"] = g.name

        if len(g) < 2:
            return g

        inds = (
            g["Industry_harmonized"]
            .dropna()
            .astype(str)
            .str.strip()
        )
        unique_inds = inds.unique()
        if len(unique_inds) < 2:
            return g

        spec_empty = _is_empty_series(g["Industry_specific"])
        if int(spec_empty.sum()) != (len(g) - 1):
            return g

        combined = _combine_industries(g["Industry_harmonized"].tolist())

        scores = g.apply(_non_empty_count_row, axis=1)
        keep_idx = scores.idxmax()

        kept = g.loc[[keep_idx]].copy()
        kept.loc[:, "Industry_harmonized"] = combined
        return kept

    out = (
        df.groupby("Entities_filtered", dropna=False, sort=False, group_keys=False)
          .apply(handle_entity, include_groups=False)
          .reset_index(drop=True)
    )
    return out

before_rows = len(duplicates_full)
duplicates_full = dedup_case_2(duplicates_full)
print(f"Rows removed (Case 2): {before_rows - len(duplicates_full)}")
print(f"Rows remaining: {len(duplicates_full)}")
print(
    "Entities appearing more than once:",
    (duplicates_full["Entities_filtered"].value_counts() > 1).sum()
)
duplicates_full[
    duplicates_full["Entities_filtered"].duplicated(keep=False)
].sort_values("Entities_filtered")

Rows removed (Case 2): 37
Rows remaining: 361
Entities appearing more than once: 168


,Industry_specific,Role,Source_dataset,Workplace,Industry_harmonized,Entities_filtered
121,NaN,NaN,codebook,NaN,Energy Industry,Alan Jeffers
122,"Public Affairs, Media Relations",Retired communicator at Exxon Mobil Corporation,target_df_updated,"ExxonMobil Corporation,Texas ,USA",NaN,Alan Jeffers
197,Climate and Sustainability,Managing Director at Campus Climate Network,target_df_updated,Campus Climate Network,NaN,Alicia Colomer
196,NaN,NaN,codebook,NaN,NGO/Thinktank/Foundation,Alicia Colomer
193,"Media Relations, Public Affairs",Senior Media Relations Manager at the American...,target_df_updated,Washington DC-Baltimore Area,NaN,Andrea Woods
...,...,...,...,...,...,...
100,solar physics and his skepticism regarding hum...,"Astrophysicist, Researcher",target_df_updated,Center for Astrophysics,NaN,Willie Soon
357,NaN,NaN,codebook,NaN,Politician,Yvonne Brathwaite Burke
358,"Politics, Civil Rights, Transportation","Former U.S. Congresswoman, Attorney",target_df_updated,"U.S. House of Representatives, Amtrak",NaN,Yvonne Brathwaite Burke
187,NaN,NaN,codebook,NaN,Journalist,Zack Budryk


In [10]:
reference_list = [
   
    "Government Agency",
    "Politician",
    "Legislation",
    "International Organisation",
    "International Climate Governance",
  
    "Academia",
    "Lobby",
    "NGO/Thinktank/Foundation",
    
    "Energy Industry",
    "Other Industry",
    "Consultancy",
    "PR Company",
   
    "Litigation",
    "Courts",
    "Legal Professionals",
 
    "News Outlet",
    "Journalist",
  
    "Keyword Climate/Energy",
    "Keyword Greenwashing",
]

In [11]:
current_values = set(
    duplicates_full["Industry_harmonized"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

reference_values = set(reference_list)

print("In data but NOT in reference list:")
print(sorted(current_values - reference_values))

print("\nIn reference list but NOT in data:")
print(sorted(reference_values - current_values))


In data but NOT in reference list:
['Academia; Consultancy', 'Academia; Government Agency', 'Consultancy; Government Agency', 'Consultancy; Legal Professionals', 'Energy Industry; Government Agency; Politician', 'Government Agency; Politician', 'International Organisation; Legal Professionals', 'Journalist; NGO/Thinktank/Foundation', 'Legal Professionals; NGO/Thinktank/Foundation']

In reference list but NOT in data:
['Courts', 'International Climate Governance', 'Keyword Obstruction', 'Legal Professionals', 'Legislation', 'Litigation', 'News Outlet']


In [12]:
column_order = [
    "Entities_filtered",
    "Industry_harmonized",
    "Industry_specific",
    "Role",
    "Workplace",
    "source_dataset",
]

duplicates_full = duplicates_full.reindex(columns=column_order)

duplicates_full.to_excel("overlap_duplicates_185.xlsx", index=False)
print("Saved: overlap_duplicates_185.xlsx")

Saved: overlap_duplicates_185.xlsx


## Now move the finalized "person" rows back into the entire codebook, append or replace 

In [13]:
# 1) rename in codebook to match duplicates_full
duplicates_full = duplicates_full.rename(columns={"Industry_harmonized": "Sub_category"})
str_cols = codebook.select_dtypes(include="object").columns
codebook[str_cols] = codebook[str_cols].apply(lambda c: c.str.strip())

# 2) counts BEFORE changes
existing = set(codebook["Entities_filtered"].astype(str).str.strip())
incoming = set(duplicates_full["Entities_filtered"].astype(str).str.strip())

n_replaced = len(existing & incoming)
n_added = len(incoming - existing)

# 3) build a 2-col incoming frame directly (no separate "updates" variable)
incoming_df = duplicates_full[["Entities_filtered", "Sub_category"]].copy()
incoming_df["Entities_filtered"] = incoming_df["Entities_filtered"].astype(str).str.strip()

# if duplicates_full has repeated Entities_filtered, keep the last one (prevents assignment issues)
incoming_df = incoming_df.drop_duplicates(subset=["Entities_filtered"], keep="last")

# 4) UPSERT: replace matches, append new
codebook = codebook.set_index("Entities_filtered")
incoming_df = incoming_df.set_index("Entities_filtered")

codebook.update(incoming_df)  # replaces Industry_harmonized where keys overlap
codebook = pd.concat([codebook, incoming_df.loc[incoming_df.index.difference(codebook.index)]])
codebook = codebook.reset_index()

print(f"Rows added: {n_added}")
print(f"Rows replaced: {n_replaced}")


NameError: name 'codebook' is not defined

In [243]:
codebook = codebook.drop_duplicates()
def _join_unique_industries(s: pd.Series) -> str:
    vals = []
    for v in s.dropna().astype(str):
        # allow already-combined strings
        parts = [p.strip() for p in v.split(";") if p.strip()]
        vals.extend(parts)
    # unique preserve order
    seen = set()
    out = []
    for x in vals:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return "; ".join(out)

# Keep other columns by taking the first non-null value per group,
# and set flag to max (so if any row had 1, the collapsed row has 1).
agg = {}
for c in codebook.columns:
    if c == "Sub_category":
        agg[c] = _join_unique_industries
    elif c == "Flag_Individuals_Codebook":
        agg[c] = "max"
    elif c == "Entities_filtered":
        continue
    else:
        agg[c] = "first"

codebook = (
    codebook
    .groupby("Entities_filtered", dropna=False, as_index=False)
    .agg(agg)
)

### Now add a flag Flag_Individuals_Codebook

In [244]:
# --- Flag rows that come from duplicates_full ---
incoming_names = (
    duplicates_full["Entities_filtered"]
    .astype(str).str.strip()
    .dropna()
    .unique()
)

codebook["Entities_filtered"] = codebook["Entities_filtered"].astype(str).str.strip()

codebook["Flag_Individuals_Codebook"] = codebook["Entities_filtered"].isin(incoming_names).astype(int)

# --- Checks: counts ---
n_flag_1 = int((codebook["Flag_Individuals_Codebook"] == 1).sum())
n_flag_0 = int((codebook["Flag_Individuals_Codebook"] == 0).sum())

print(f"Flag=1 count: {n_flag_1}")
print(f"Flag=0 count: {n_flag_0}")

# check expected split: 177 + 8 = 185
print(f"Check (expect 185): {n_flag_1} == {177 + 8} -> {n_flag_1 == (177 + 8)}")

# --- Check: no duplicate names in codebook ---
dup_count = int(codebook["Entities_filtered"].duplicated().sum())
print(f"Duplicate Entities_filtered rows in codebook: {dup_count}")

if dup_count > 0:
    print("Duplicates (first few):")
    print(codebook.loc[codebook["Entities_filtered"].duplicated(keep=False), "Entities_filtered"]
          .value_counts()
          .head(10))

print("Flag_Individuals_Codebook present:",
      "Flag_Individuals_Codebook" in codebook.columns)

Flag=1 count: 185
Flag=0 count: 797
Check (expect 185): 185 == 185 -> True
Duplicate Entities_filtered rows in codebook: 0
Flag_Individuals_Codebook present: True


In [245]:
print(sorted(codebook["Sub_category"].dropna().unique()))

current_values = set(
    codebook["Sub_category"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

reference_values = set(reference_list)

print("In current codebook but NOT in list from Table 6:")
print(sorted(current_values - reference_values))

print("\nIn list from Table 6 but NOT in current codebook:")
print(sorted(reference_values - current_values))
print(codebook.columns.tolist())

['', 'Academia', 'Academia; Consultancy', 'Academia; Government Agency', 'Consultancy', 'Consultancy; Government Agency', 'Consultancy; Legal Professionals', 'Courts', 'Energy Industry', 'Energy Industry; Government Agency; Politician', 'Energy Industry; Lobby', 'Government Agency', 'Government Agency; Litigation', 'Government Agency; Politician', 'International Climate Governance', 'International Organisation', 'International Organisation; Legal Professionals', 'Journalist', 'Journalist; NGO/Thinktank/Foundation', 'Keyword Climate Environment', 'Keyword Climate Environment; Legislation', 'Keyword Climate Environment; Litigation', 'Keyword Obstruction', 'Legal Professionals', 'Legal Professionals; NGO/Thinktank/Foundation', 'Legislation', 'Litigation', 'Lobby', 'NGO/Thinktank/Foundation', 'News Outlet', 'Other Industry', 'PR Company', 'Politician']
In current codebook but NOT in list from Table 6:
['', 'Academia; Consultancy', 'Academia; Government Agency', 'Consultancy; Government Age

In [247]:
print(f"Rows in final NER Codebook: {len(codebook)}")
print(codebook.columns.tolist())
n_flag_1 = int((codebook["Flag_Individuals_Codebook"] == 1).sum())
n_flag_0 = int((codebook["Flag_Individuals_Codebook"] == 0).sum())

print(f"Flag=1 count: {n_flag_1}")
print(f"Flag=0 count: {n_flag_0}")

Rows in final NER Codebook: 982
['Entities_filtered', 'Sub_category', 'Flag_Individuals_Codebook']
Flag=1 count: 185
Flag=0 count: 797


In [248]:
codebook.to_excel("NER_Codebook_982.xlsx", index=False)
print("Saved: NER_Codebook_982.xlsx")

Saved: NER_Codebook_982.xlsx


In [250]:
# split multi-valued sub_category rows and explode
subcat_counts = (
    codebook["Sub_category"]
    .dropna()
    .astype(str)
    .str.split(";")
    .explode()
    .str.strip()
    .value_counts()
    .reset_index()
)

subcat_counts.columns = ["Sub_category", "count"]

print(subcat_counts)

#6 people weirdly do not have any affiliation, but the _185 dataset has them.

                        Sub_category  count
0                           Academia    126
1                    Energy Industry    106
2                  Government Agency    104
3                         Politician     95
4        Keyword Climate Environment     86
5           NGO/Thinktank/Foundation     69
6                              Lobby     68
7                         Litigation     45
8                             Courts     42
9                Legal Professionals     40
10                    Other Industry     39
11                       Legislation     37
12                       Consultancy     34
13                       News Outlet     33
14        International Organisation     18
15                        Journalist     17
16                        PR Company     13
17               Keyword Obstruction     12
18  International Climate Governance     11
19                                        7


In [252]:
print("Sum of counts:", subcat_counts["count"].sum())

Sum of counts: 1002


### Append the 700 entities from the external news corpus: LexisNexis_Entities.xlsx - check for duplicates

In [22]:
ln_path = Path(r"C:\Users\charlott\Dropbox (Personal)\Paper_Corporate_Obstructionism\Data\LexisNexis\LexisNexis_Entities.xlsx")

lexisnexis = pd.read_excel(ln_path)
print("ln_path columns:", lexisnexis.columns.tolist())
print("Rows:", len(lexisnexis))

ln_path columns: ['Entities_filtered', 'Sub_category']
Rows: 700


In [90]:
codebook_path = Path(r"C:\Users\charlott\Dropbox (Personal)\CSSN_Team_Folder\Data\NER_Codebook_982.xlsx")

codebook = pd.read_excel(codebook_path)
print("codebook_path columns:", codebook.columns.tolist())
print("Rows:", len(codebook))


codebook_path columns: ['Entities_filtered', 'Sub_category', 'Flag_Individuals_Codebook']
Rows: 982


In [91]:
# normalize keys (important to avoid false mismatches)
codebook["Entities_filtered"] = codebook["Entities_filtered"].astype(str).str.strip()
lexisnexis["Entities_filtered"] = lexisnexis["Entities_filtered"].astype(str).str.strip()

# build set for fast lookup
codebook_entities = set(codebook["Entities_filtered"])

# anti-join
lexis_only = lexisnexis[
    ~lexisnexis["Entities_filtered"].isin(codebook_entities)
]

# print first 50
print(lexis_only.head(300))

# optional: how many total
print("Total new entities from Lexisnexis that are not in the codebook yet:", len(lexis_only))

                                   Entities_filtered  \
0                                           Amazonia   
1                                                Ann   
2                           Accountability Committee   
3                             Advancement of climate   
4                                     Alexander Wood   
..                                               ...   
280  Global Climate Coalition Communications Program   
281                Accountability for Climate Change   
283                           Denial, Disinformation   
286                                    Denier Groups   
290                                Reputational Risk   

                    Sub_category  
0                 Other Industry  
1                 Other Industry  
2    Keyword Climate Environment  
3    Keyword Climate Environment  
4                            NaN  
..                           ...  
280  Keyword Climate Environment  
281  Keyword Climate Environment  
283         

In [92]:
lexis_only.to_excel("lexis_only_302.xlsx", index=False)
print("Saved: lexis_only_302.xlsx")

Saved: lexis_only_302.xlsx


In [93]:
# Ran an AI over it (just a bunch of regex rules in python), adding confidence column, then went through it manually, corrected some classifications and removed rows that we do not need.

In [94]:
ln_path = Path(r"C:\Users\charlott\Dropbox (Personal)\Paper_Corporate_Obstructionism\Data\LexisNexis\lexis_only_302_classified.xlsx")

lexisnexis = pd.read_excel(ln_path)
print("ln_path columns:", lexisnexis.columns.tolist())
print("Rows:", len(lexisnexis))

# Normalize Entities_filtered
lexisnexis["Entities_filtered"] = (
    lexisnexis["Entities_filtered"]
    .astype(str)
    .str.strip()
    .str.replace(r"^the\s+", "", regex=True, flags=re.IGNORECASE)   # remove leading "the "
    .str.replace(r"(?:'s|’s)\s*$", "", regex=True)                   # remove trailing 's or ’s
    .str.strip()
)

# Drop empty after cleaning
lexisnexis = lexisnexis[lexisnexis["Entities_filtered"] != ""]

# Deduplicate
lexisnexis = lexisnexis.drop_duplicates(subset=["Entities_filtered"])

print("Rows after cleaning & dedup:", len(lexisnexis))
print("Rows after drop:", len(lexisnexis))

ln_path columns: ['Entities_filtered', 'Sub_category', 'Confidence']
Rows: 302
Rows after cleaning & dedup: 223
Rows after drop: 223


### now add entities that are not in codebook to codebook (adds 170 rows)

In [95]:
# Drop "Confidence" column from lexisnexis 
lexisnexis = lexisnexis.drop(columns=["Confidence"], errors="ignore")
lexisnexis["Entities_filtered"] = (
    lexisnexis["Entities_filtered"]
    .astype(str)
    .str.strip()
    .apply(lambda x: x.title() if x.isupper() else x)
)

# Identify new entities (not already in codebook)
new_entities = lexisnexis[
    ~lexisnexis["Entities_filtered"].isin(codebook["Entities_filtered"])
]

# Append them to codebook
codebook = pd.concat([codebook, new_entities], ignore_index=True)

print("New rows added:", len(new_entities))
print("Total rows in codebook:", len(codebook))

empty_subcat_count = (
    new_entities["Sub_category"].isna() |
    (new_entities["Sub_category"].astype(str).str.strip() == "")
).sum()

print("New rows with empty Sub_category:", empty_subcat_count)

New rows added: 169
Total rows in codebook: 1151
New rows with empty Sub_category: 34


### Now add the info about the acronyms 


In [96]:
acronyms = pd.read_excel(
    r"C:\Users\charlott\Dropbox (Personal)\Paper_Corporate_Obstructionism\Analysis\B_NER\Preparation_Database_Publication\ner_acronyms_v2_cleaned.xlsx"
)

acronyms = acronyms.drop(columns=["MISC explain."], errors="ignore")
print("acronyms columns:", acronyms.columns.tolist())
print("codebook columns:", codebook.columns.tolist())

codebook = codebook[codebook["Entities_filtered"] != "ICN"].reset_index(drop=True)

# Merge strictly on Entities_filtered
merged = codebook.merge(
    acronyms[["Entities_filtered", "Full_name"]],
    on="Entities_filtered",
    how="left",
    indicator=True
)

# Count how many rows received appended values
altered_rows = (merged["_merge"] == "both").sum()
print("Number of rows altered (identical match):", altered_rows)

# Entities in acronyms but NOT in codebook
missing_entities = acronyms.loc[
    ~acronyms["Entities_filtered"].isin(codebook["Entities_filtered"]),
    "Entities_filtered"
].unique()

print("\nEntities in acronyms but not in codebook:")
print(missing_entities)
# checked 

# Drop merge indicator
codebook = merged.drop(columns="_merge")

acronyms columns: ['Entities_filtered', 'Full_name', 'Flag_drop']
codebook columns: ['Entities_filtered', 'Sub_category', 'Flag_Individuals_Codebook']
Number of rows altered (identical match): 68

Entities in acronyms but not in codebook:
['BTU' 'NAMCS' 'MMBTU' 'COVID' 'ICN' 'NBC' 'PBS']


In [97]:
print("codebook columns:", codebook.columns.tolist())
# so now I have e.g. LSU in Entities_filtered Lousiana State University in Full_Name. I now want to write LSU; Lousiana State University in Entities_filtered and then drop the full name column

codebook["Entities_filtered"] = codebook["Entities_filtered"].astype(str).str.strip()
codebook["Full_name"] = codebook["Full_name"].astype(str).str.strip()

# Create combined column only where Full_name exists
codebook["Entities_filtered"] = np.where(
    codebook["Full_name"].notna() & (codebook["Full_name"] != "") & (codebook["Full_name"] != "nan"),
    codebook["Entities_filtered"] + "; " + codebook["Full_name"],
    codebook["Entities_filtered"]
)

# Drop Full_name column
codebook = codebook.drop(columns="Full_name")

codebook columns: ['Entities_filtered', 'Sub_category', 'Flag_Individuals_Codebook', 'Full_name']


In [98]:
# check how many Entities_filtered have a ";" - these are the acronyms 
num_acronyms = codebook["Entities_filtered"].str.contains(";", na=False).sum()
print("Number of Entities_filtered containing ';' (acronyms):", num_acronyms)
# nice, 69 as expected 

Number of Entities_filtered containing ';' (acronyms): 68


In [99]:
# make sure these are all Legislations
prefixes = ["BTU", "CBOR", "FCRS", "DACA", "GOMESA", "ESA", "NAFTA", "NEPA"]
pattern = r'^(' + '|'.join(prefixes) + ')'
filtered = codebook[codebook["Entities_filtered"].str.startswith(tuple(prefixes), na=False)]
print(filtered[["Entities_filtered", "Sub_category"]])

                                Entities_filtered  \
101                CBOR; Community Bill of Rights   
216  DACA; Deferred Action for Childhood Arrivals   
256          ESA; Environmentally Sensitive Areas   
257                   ESA; Endangered Species Act   
294          FCRS; Fuel Carbon Reduction Standard   
318    GOMESA; Gulf of Mexico Energy Security Act   
518    NAFTA; North American Free Trade Agreement   
522       NEPA; National Environmental Policy Act   

                                 Sub_category  
101                               Legislation  
216                               Legislation  
256  Keyword Climate Environment; Legislation  
257  Keyword Climate Environment; Legislation  
294                               Legislation  
318                               Legislation  
518                               Legislation  
522                               Legislation  


In [100]:
# make sure these are all lobby or legislation
prefixes = [
    "BTU", "CBOR", "FCRS", "DACA", "GOMESA", "ESA", "NAFTA", "NEPA",
    "AIGN", "APPEA", "AIP", "BDI", "BCA", "CAPP", "CBI", "CEFIC",
    "ERT", "IOGP", "IETA", "IGU", "OGUK", "OGCI", "VCI",
    "AOP", "VNPI", "AAAS", "NAM"
]

# Filter rows where Entities_filtered starts with any prefix
filtered = codebook[
    codebook["Entities_filtered"].str.startswith(tuple(prefixes), na=False)
]

# Print entity and sub category
print(filtered[["Entities_filtered", "Sub_category"]])


                                     Entities_filtered  \
6    AAAS; American Association for the Advancement...   
8         AIGN; Australian Industry Greenhouse Network   
9               AIP; Australian Institute of Petroleum   
12    AOP; Association of Petroleum Products Operators   
16   APPEA; Australian Petroleum Production & Explo...   
58                  BCA; Business Council of Australia   
59          BDI; Bundesverband der Deutschen Industrie   
99   CAPP; Canadian Association of Petroleum Producers   
100             CBI; Confederation of British Industry   
101                     CBOR; Community Bill of Rights   
107          CEFIC; European Chemical Industry Council   
216       DACA; Deferred Action for Childhood Arrivals   
255        ERT; European Round Table of Industrialists   
256               ESA; Environmentally Sensitive Areas   
257                        ESA; Endangered Species Act   
294               FCRS; Fuel Carbon Reduction Standard   
318         GO

In [105]:
# manually add 
new_entries = pd.DataFrame({
    "Entities_filtered": ["ICN; Inside Climate News Media", "NBC; National Broadcasting Co", "PBS; Public Broadcasting Service"],
    "Sub_category": ["Media; Keyword Climate/Energy", "Media", "Media"]
})
new_entries = new_entries.reindex(columns=codebook.columns)
codebook = pd.concat([codebook, new_entries], ignore_index=True)

# make sure these are media
prefixes = [
    "ICN", "NBC", "PBS"
]

# Filter rows where Entities_filtered starts with any prefix
filtered = codebook[
    codebook["Entities_filtered"].str.startswith(tuple(prefixes), na=False)
]

# Print entity and sub category
print(filtered[["Entities_filtered", "Sub_category"]])

                     Entities_filtered                        Sub_category
1151    ICN; Inside Climate News Media  Media; Keyword Climate Environment
1152     NBC; National Broadcasting Co                               Media
1153  PBS; Public Broadcasting Service                               Media


In [149]:
# merge with this file, where we manually collect meta data for each individual (as above but finalized/completed, by Josh)

filename = r"C:\Users\charlott\Dropbox (Personal)\CSSN_Team_Folder\Data\NER_email_extraction\Extracted_email_names_meta_info.xlsx"
meta_info = pd.read_excel(filename)
print("Rows:", len(meta_info))
print(meta_info.columns.tolist())
dup_fullname_rows = meta_info["Entities_filtered"].duplicated().sum()
print("Duplicate rows in df (Entities_filtered):", dup_fullname_rows)

Rows: 555
['Entities_filtered', 'Role', 'Employee_level', 'Workplace', 'Industry_broad', 'Industry_specific']
Duplicate rows in df (Entities_filtered): 0


In [150]:
# Append Employee_level to Role in parentheses (if non-empty)
meta_info['Role'] = meta_info.apply(
    lambda row: (
        (str(row['Role']).strip() if pd.notna(row['Role']) else '') +
        (f" ({str(row['Employee_level']).strip()}-level employee)"
         if pd.notna(row['Employee_level']) and str(row['Employee_level']).strip() != ''
         else '')
    ),
    axis=1
)

# Drop Employee_level column
meta_info.drop(columns=['Employee_level'], inplace=True)

# Print 2 examples where Role ends with ")"
examples = meta_info[meta_info['Role'].str.endswith(")", na=False)].head(4)
print(examples[['Role']])

                                                 Role
14  U.S. academic, business executive, and author,...
65  Vice President of Government Relations (High-l...
66  Senior Manager of Media Relations and Rapid Re...
67  Senior VP of Communications (High-level employee)


In [151]:
print(codebook.columns.tolist())

['Entities_filtered', 'Sub_category', 'Flag_Individuals_Codebook']


In [152]:
# some cleaning before we merge 
meta_info = meta_info.applymap(lambda x: x.strip() if isinstance(x, str) else x)
meta_info['Industry_broad'] = meta_info['Industry_broad'].replace({
    "International Organization": "International Organisation",
    "Energy/Industry": "Energy Industry",
    "Academia,Politician": "Academia; Politician"
})

print(meta_info['Industry_broad'].dropna().unique())
meta_info = meta_info.rename(columns={"Industry_broad": "Sub_category"})

['Consultancy' 'Academia' 'Academia; Politician' 'Energy Industry'
 'Government Agency' 'International Organisation' 'Journalist'
 'Keyword Climate Environment' 'Litigation' 'Lobby' 'Media' 'News Outlet'
 'NGO/Thinktank/Foundation' 'Other Industry' 'Politician' 'PR Company']


C:\Users\charlott\AppData\Local\Temp\ipykernel_24300\1748258121.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  meta_info = meta_info.applymap(lambda x: x.strip() if isinstance(x, str) else x)


In [153]:
# Step 1: Find identical columns
common_columns = meta_info.columns.intersection(codebook.columns)
print("Columns present in both meta_info and codebook:")
print(common_columns.tolist())

# Step 2: Check if all values in Sub_category in meta_info are in codebook
missing_values = set(meta_info['Sub_category'].dropna()) - set(codebook['Sub_category'].dropna())

if not missing_values:
    print("All values in Sub_category in meta_info are contained in codebook.")
else:
    print("Values in Sub_category in meta_info not found in codebook:")
    print(missing_values)


Columns present in both meta_info and codebook:
['Entities_filtered', 'Sub_category']
Values in Sub_category in meta_info not found in codebook:
{'Academia; Politician'}


In [155]:
# Ensure case-insensitive comparison
meta_entities = meta_info[['Entities_filtered', 'Sub_category']].copy()
meta_entities['Entities_lower'] = meta_entities['Entities_filtered'].str.lower()

code_entities_lower = codebook['Entities_filtered'].str.lower()

# Find rows in meta_info not yet in codebook (case-insensitive)
missing_rows = meta_entities[~meta_entities['Entities_lower'].isin(code_entities_lower)].copy()

# Prepare new rows to append
new_rows = missing_rows[['Entities_filtered', 'Sub_category']].copy()
new_rows['Flag_Individuals_Codebook'] = 1

# Reorder columns to match codebook
new_rows = new_rows[codebook.columns]

# Append to codebook
codebook = pd.concat([codebook, new_rows], ignore_index=True)

print(f"Appended {len(new_rows)} new rows to codebook.")

Appended 364 new rows to codebook.


In [158]:
new_entities_sample = new_rows[['Entities_filtered', 'Flag_Individuals_Codebook']].sample(20, random_state=42)

print(new_entities_sample)

      Entities_filtered  Flag_Individuals_Codebook
256        Robert Miner                          1
96          Henry Cheng                          1
78       Melanie Rogers                          1
451     Daphne Magnuson                          1
120      Betsey Weltner                          1
246    Paulette Cousino                          1
139           Dana Wood                          1
182        Jamie Graves                          1
215       Maria Amezaga                          1
189         John Harvey                          1
290       Tyrone Kaljee                          1
102   Alejandra Castano                          1
387   AntÃ³nio Guterres                          1
118         Ben Mathews                          1
200        Keith Botley                          1
369    Catherine Landry                          1
5          Susanne Rust                          1
288            Tom Wolf                          1
424  Will Riddell-McKay        

In [166]:
print(f"Final length of codebook {len(codebook)}")
print(f"Number of individuals inside {codebook['Flag_Individuals_Codebook'].sum()}")

Final length of codebook 1518
Number of individuals inside 549.0


In [163]:
empty_subcat_count = codebook[
    (codebook['Flag_Individuals_Codebook'] == 1) &
    (codebook['Sub_category'].isna() | (codebook['Sub_category'].str.strip() == ''))
].shape[0]

print(f"Number of rows with Flag_Individuals_Codebook = 1 and empty Sub_category: {empty_subcat_count}")

Number of rows with Flag_Individuals_Codebook = 1 and empty Sub_category: 0


In [170]:
output_path = r"C:\Users\charlott\Dropbox (Personal)\CSSN_Team_Folder\github_upload_NER\NER_Codebook_1518.xlsx"
codebook.to_excel(output_path, index=False)
print("Saved: NER_Codebook_1518.xlsx")

Saved: NER_Codebook_1518.xlsx


In [168]:
# Done.